In [1]:
import torch
from PIL import Image
import open_clip

/Users/karlxu/workspace/github/xuyuji9000/machine-learning-playground/02_models/01_clip/01_open-clip-example/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load the model and image preprocessing pipeline
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
model.eval()  # Set model to evaluation mode


CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768

In [3]:
# 2. Get the corresponding tokenizer
tokenizer = open_clip.get_tokenizer('ViT-B-32')

In [4]:
# 3. Preprocess the image
# Replace 'CLIP.png' with the path to your image file
image = preprocess(Image.open('CLIP.png')).unsqueeze(0)

In [5]:
# 4. Tokenize text labels
text = tokenizer(['a diagram', 'a dog', 'a cat'])
# text[0]

In [ ]:
# 5. Extract features and compute similarities
with torch.no_grad(), torch.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
  image_features = model.encode_image(image)
  text_features = model.encode_text(text)

  # Normalize features
  image_features /= image_features.norm(dim=-1, keepdim=True)
  text_features /= text_features.norm(dim=-1, keepdim=True)

  # Calculate cosine similarity and apply softmax to get probabilities
  text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)



tensor([[1.0000e+00, 5.5313e-04, 1.3161e-04]], dtype=torch.bfloat16)

In [15]:
# 6. Print results
print('Label probabilities:', text_probs.float().cpu().numpy())

Label probabilities: [[1.0000000e+00 5.5313110e-04 1.3160706e-04]]
